In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
import lightgbm as lgb
from transformers import AutoModel, AutoTokenizer
import os

In [2]:
desc_tensor = torch.load("torch_data/descriptors.pt")

In [3]:
print(len(desc_tensor))
print(desc_tensor)

16648
tensor([[ 0.2666,  0.2666, -1.0064,  ..., -0.2238, -0.1981, -0.0439],
        [ 0.4235,  0.4235, -0.3401,  ..., -0.2238, -0.1981, -0.0439],
        [ 0.4412,  0.4412, -0.9842,  ..., -0.2238, -0.1981, -0.0439],
        ...,
        [ 0.7704,  0.7704, -0.1347,  ..., -0.2238, -0.1981, -0.0439],
        [ 0.7704,  0.7704, -0.1347,  ..., -0.2238, -0.1981, -0.0439],
        [ 0.7704,  0.7704, -0.1347,  ..., -0.2238, -0.1981, -0.0439]])


In [4]:
msl_new = pd.read_csv('input_data/msl_new.csv')
print(len(msl_new))
msl_new.head()

16648


,Chromophore,Solvent,Quantum yield
0,O=C([O-])c1ccccc1-c1c2ccc(=O)cc-2oc2cc([O-])ccc12,O,0.950
1,O=C([O-])c1ccccc1C1=c2cc3c4c(c2Oc2c1cc1c5c2CCC...,CO,1.000
2,O=C([O-])c1ccccc1-c1c2cc(Br)c(=O)c(Br)c-2oc2c(...,O,0.200
3,O=C([O-])c1ccccc1-c1c2cc(I)c(=O)c(I)c-2oc2c(I)...,O,0.020
4,O=C([O-])c1c(Cl)c(Cl)c(Cl)c(Cl)c1-c1c2cc(I)c(=...,O,0.018


In [7]:
vecs_mols = torch.load("embedding_caches/mols_new.pt", map_location="cpu")
print(vecs_mols.shape)

vecs_sols = torch.load("embedding_caches/sols_new.pt", map_location="cpu")
print(vecs_sols.shape)


torch.Size([16648, 256])
torch.Size([16648, 256])


In [8]:
X = torch.cat([vecs_mols, vecs_sols, desc_tensor], dim=1)
mean = torch.nanmean(X)
X = X.nan_to_num(mean)
print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 729])
torch.Size([16648])


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(13318, 729) (13318,)
(1665, 729) (1665,)
(1665, 729) (1665,)


In [10]:
print(X_np.shape)
print(y_np.shape)

(16648, 729)
(16648,)


In [11]:
np.save('model_data/X_desc.npy', X_np)
np.save('model_data/y_desc.npy', y_np)

In [12]:
mlp = MLPRegressor(
    hidden_layer_sizes=(512, 256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=2000,
    random_state=42,
    warm_start=False,
    early_stopping=True
)

X_train_mlp = np.vstack([X_train, X_val])
y_train_mlp = np.concatenate([y_train, y_val])

mlp.fit(X_train_mlp, y_train_mlp)

y_test_pred = mlp.predict(X_test)
print("test:")
print(f"R2: {r2_score(y_test, y_test_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.3f}")

test:
R2: 0.640
RMSE: 0.181


In [13]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist"
)

eval_set = [(X_train, y_train), (X_val, y_val)]

xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_val_xgb = xgb_model.predict(X_val)
print("validation:")
print(f"R Squared: {r2_score(y_val, y_val_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_xgb)):.3f}")
print()
y_test_xgb = xgb_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_xgb)):.3f}")

validation:
R Squared: 0.657
RMSE: 0.182

test:
R Squared: 0.707
RMSE: 0.163


In [14]:
lgb_model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.1, random_state=42)

eval_set = [(X_train, y_train), (X_val, y_val)]

lgb_model.fit(X_train, y_train, eval_set=eval_set, eval_metric="rmse", callbacks=[lgb.early_stopping(stopping_rounds=50)])

y_val_lgb = lgb_model.predict(X_val)
print("validation:")
print(f"R Squared: {r2_score(y_val, y_val_lgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_lgb)):.3f}")
print()
print("test:")
y_test_lgb = lgb_model.predict(X_test)
print(f"R Squared: {r2_score(y_test, y_test_lgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_lgb)):.3f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019914 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 146850
[LightGBM] [Info] Number of data points in the train set: 13318, number of used features: 706
[LightGBM] [Info] Start training from score 0.344984
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[997]	training's rmse: 0.0347543	training's l2: 0.00120786	valid_1's rmse: 0.185881	valid_1's l2: 0.0345516
validation:
R Squared: 0.644
RMSE: 0.186

test:
R Squared: 0.698
RMSE: 0.166


/Users/utoglu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/utoglu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### Fingerprints

In [15]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator

In [16]:
fgp_tensor = torch.load('torch_data/fingerprints.pt')
print(len(fgp_tensor))
print(fgp_tensor)

16648
tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], dtype=torch.int32)


In [17]:
X = torch.cat([vecs_mols, vecs_sols, desc_tensor, fgp_tensor], dim=1)
mean = torch.nanmean(X)
X = X.nan_to_num(mean)
print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 2777])
torch.Size([16648])


In [18]:
import numpy as np
from sklearn.model_selection import train_test_split

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(13318, 2777) (13318,)
(1665, 2777) (1665,)
(1665, 2777) (1665,)


In [19]:
print(X_np.shape)
print(y_np.shape)

(16648, 2777)
(16648,)


In [20]:
np.save('model_data/X_fgp.npy', X_np)
np.save('model_data/y_gfp.npy', y_np)

In [21]:
mlp = MLPRegressor(
    hidden_layer_sizes=(512, 256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=2000,
    random_state=42,
    warm_start=False,
    early_stopping=True
)

X_train_mlp = np.vstack([X_train, X_val])
y_train_mlp = np.concatenate([y_train, y_val])

mlp.fit(X_train_mlp, y_train_mlp)

y_test_pred = mlp.predict(X_test)
print("test:")
print(f"R2: {r2_score(y_test, y_test_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.3f}")

test:
R2: 0.717
RMSE: 0.160


In [22]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_val, y_val)]

xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_val_xgb = xgb_model.predict(X_val)
print("validation:")
print(f"R Squared: {r2_score(y_val, y_val_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_xgb)):.3f}")
print()
y_test_xgb = xgb_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_xgb)):.3f}")

/Users/utoglu/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [16:23:34] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


validation:
R Squared: 0.667
RMSE: 0.180

test:
R Squared: 0.709
RMSE: 0.163


In [23]:
lgb_model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.1, random_state=42)

eval_set = [(X_val, y_val)]

lgb_model.fit(X_train, y_train, eval_set=eval_set, eval_metric="rmse", callbacks=[lgb.early_stopping(stopping_rounds=50)])

y_val_lgb = lgb_model.predict(X_val)
print("validation:")
print(f"R Squared: {r2_score(y_val, y_val_lgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_lgb)):.3f}")
print()
print("test:")
y_test_lgb = lgb_model.predict(X_test)
print(f"R Squared: {r2_score(y_test, y_test_lgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_lgb)):.3f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.131227 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 154575
[LightGBM] [Info] Number of data points in the train set: 13318, number of used features: 2574
[LightGBM] [Info] Start training from score 0.344984
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[723]	valid_0's rmse: 0.1794	valid_0's l2: 0.0321845
validation:
R Squared: 0.668
RMSE: 0.179

test:
R Squared: 0.705
RMSE: 0.164


/Users/utoglu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/utoglu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
